## Scenario 1: Retail Inventory Analysis

### Prompt:

I have a Pandas DataFrame with the columns product_id, category, stock_level, last_restock_date, sales_last_30_days, supplier_lead_time, and unit_cost. Write Python code to calculate inventory turnover for each product, identify slow-moving items, determine which products may need to be restocked soon, and create a visualization of the results.

### Generated Code:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Sample data: replace this block with your own DataFrame ---
rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "product_id": [f"P{i:04d}" for i in range(1, n + 1)],
    "category": rng.choice(["Electronics", "Apparel", "Grocery", "Home", "Toys"], n),
    "stock_level": rng.integers(0, 500, n),
    "last_restock_date": pd.Timestamp.today().normalize()
                         - pd.to_timedelta(rng.integers(1, 180, n), unit="D"),
    "sales_last_30_days": rng.poisson(rng.gamma(1.5, 40, n)),
    "supplier_lead_time": rng.integers(3, 30, n),  # days
    "unit_cost": rng.uniform(2, 250, n).round(2),
})
df.head()

In [ ]:
PERIOD_DAYS = 30           # sales window covered by sales_last_30_days
SAFETY_DAYS = 7            # buffer on top of supplier lead time
SLOW_DAYS_OF_SUPPLY = 90   # more than 90 days of stock on hand counts as slow-moving
TODAY = pd.Timestamp.today().normalize()

inv = df.copy()
inv["last_restock_date"] = pd.to_datetime(inv["last_restock_date"])
inv["daily_sales"] = inv["sales_last_30_days"] / PERIOD_DAYS
inv["inventory_value"] = inv["stock_level"] * inv["unit_cost"]
inv["days_since_restock"] = (TODAY - inv["last_restock_date"]).dt.days

# 1. Inventory turnover = COGS / average inventory value.
# Only current stock is known, so opening stock is estimated as current stock + units sold
# (this ignores any restock inside the 30-day window).
cogs = inv["sales_last_30_days"] * inv["unit_cost"]
avg_inventory_value = (inv["stock_level"] + inv["sales_last_30_days"] / 2) * inv["unit_cost"]
inv["turnover_30d"] = (cogs / avg_inventory_value).fillna(0)
inv["turnover_annual"] = inv["turnover_30d"] * 365 / PERIOD_DAYS

# Days of supply: how long current stock lasts at the current sales rate.
# No sales + stock on hand = never sells out (inf); no sales + no stock = 0.
inv["days_of_supply"] = np.where(
    inv["daily_sales"] > 0,
    inv["stock_level"] / inv["daily_sales"].replace(0, np.nan),
    np.where(inv["stock_level"] > 0, np.inf, 0),
)

# 2. Slow-moving items: stock that would take more than 90 days to sell.
inv["slow_moving"] = inv["days_of_supply"] > SLOW_DAYS_OF_SUPPLY

# 3. Restock needs: reorder point = expected demand during lead time + safety buffer.
inv["reorder_point"] = inv["daily_sales"] * (inv["supplier_lead_time"] + SAFETY_DAYS)
inv["suggested_order_qty"] = np.ceil(
    (inv["daily_sales"] * (inv["supplier_lead_time"] + SAFETY_DAYS + PERIOD_DAYS)
     - inv["stock_level"]).clip(lower=0)
).astype(int)

selling = inv["daily_sales"] > 0
urgent = selling & (inv["days_of_supply"] < inv["supplier_lead_time"])  # stocks out before an order could arrive
soon = selling & (inv["stock_level"] <= inv["reorder_point"])           # within the safety buffer
inv["status"] = np.select(
    [urgent, soon, inv["slow_moving"]],
    ["Urgent restock", "Restock soon", "Slow-moving"],
    default="Healthy",
)

# --- Results ---
print(inv["status"].value_counts().to_string(), "\n")

print("Median annual turnover by category:")
print(inv.groupby("category")["turnover_annual"].median().sort_values().round(2).to_string(), "\n")

slow_items = (inv[inv["slow_moving"]]
              .sort_values("inventory_value", ascending=False)
              [["product_id", "category", "stock_level", "sales_last_30_days",
                "days_of_supply", "turnover_annual", "inventory_value", "days_since_restock"]])
print(f"Slow-moving items: {len(slow_items)}, "
      f"${slow_items['inventory_value'].sum():,.0f} of inventory tied up")
display(slow_items.head(10).round(1))

restock = (inv[inv["status"].isin(["Urgent restock", "Restock soon"])]
           .assign(days_buffer=lambda d: d["days_of_supply"] - d["supplier_lead_time"])
           .sort_values("days_buffer")
           [["product_id", "category", "status", "stock_level", "days_of_supply",
             "supplier_lead_time", "reorder_point", "suggested_order_qty"]])
print(f"Products to restock: {len(restock)}")
display(restock.head(15).round(1))

In [ ]:
STATUS_COLORS = {
    "Urgent restock": "#d03b3b",
    "Restock soon": "#fab219",
    "Slow-moving": "#898781",
    "Healthy": "#0ca30c",
}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
DOS_CAP = 180  # cap days of supply on the chart so "never sells out" (inf) still shows

plt.rcParams.update({
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
ax1, ax2, ax3, ax4 = axes.flat

# (a) Days of supply vs supplier lead time, colored by status
for status, color in STATUS_COLORS.items():
    d = inv[inv["status"] == status]
    ax1.scatter(d["supplier_lead_time"], d["days_of_supply"].clip(upper=DOS_CAP),
                s=36, color=color, edgecolor="white", linewidth=0.8, label=status)
lt = np.array([0, inv["supplier_lead_time"].max() + 2])
ax1.plot(lt, lt, color=INK, lw=1, ls="--")
ax1.text(lt[1], lt[1] + 4, "stockout before\nreorder arrives", ha="right", va="bottom", fontsize=8, color=MUTED)
ax1.axhline(SLOW_DAYS_OF_SUPPLY, color=MUTED, lw=1, ls=":")
ax1.text(lt[1], SLOW_DAYS_OF_SUPPLY + 3, f"slow-moving (> {SLOW_DAYS_OF_SUPPLY} days)",
         ha="right", fontsize=8, color=MUTED)
ax1.set(xlabel="Supplier lead time (days)", ylabel=f"Days of supply (capped at {DOS_CAP})",
        title="Days of supply vs. supplier lead time")
ax1.grid(color=GRID, lw=0.6)
ax1.legend(frameon=False, loc="upper left")

# (b) Median annual turnover by category
turn = inv.groupby("category")["turnover_annual"].median().sort_values()
ax2.barh(turn.index, turn.values, color="#2a78d6", height=0.6)
for y, v in enumerate(turn.values):
    ax2.text(v, y, f" {v:.1f}×", va="center", color=INK)
ax2.set(xlabel="Median inventory turnover (annualized)", title="Inventory turnover by category")
ax2.grid(axis="x", color=GRID, lw=0.6)

# (c) Inventory value by category, split by status
value = (inv.pivot_table(index="category", columns="status", values="inventory_value",
                         aggfunc="sum", fill_value=0)
            .reindex(columns=list(STATUS_COLORS), fill_value=0)
            .loc[turn.index])
left = np.zeros(len(value))
for status, color in STATUS_COLORS.items():
    ax3.barh(value.index, value[status] / 1000, left=left, color=color,
             edgecolor="white", linewidth=1.5, height=0.6, label=status)
    left += value[status].values / 1000
ax3.set(xlabel="Inventory value ($ thousands)", title="Where inventory value sits")
ax3.grid(axis="x", color=GRID, lw=0.6)
ax3.legend(frameon=False, loc="lower right", fontsize=8)

# (d) Top 15 restock priorities: days of supply vs lead time
top = restock.head(15).iloc[::-1]
ax4.barh(top["product_id"], top["days_of_supply"], height=0.6,
         color=top["status"].map(STATUS_COLORS))
ax4.scatter(top["supplier_lead_time"], top["product_id"], marker="|", s=250, color=INK,
            linewidth=2, label="Supplier lead time", zorder=3)
ax4.set(xlabel="Days", title="Top 15 restock priorities (bar = days of supply left)")
ax4.grid(axis="x", color=GRID, lw=0.6)
ax4.legend(frameon=False, loc="lower right")

fig.suptitle("Retail inventory health", fontsize=15, fontweight="bold", x=0.01, ha="left", color=INK)
fig.tight_layout()
plt.show()

### Follow-up Prompts:

Can you review the code you generated and make it more robust? I want it to validate the required columns, handle missing or invalid values, deal with bad dates and negative numbers, and avoid calculation errors. Please also explain why you chose the 90-day slow-moving threshold and 7-day safety buffer, and make those values configurable. Double-check the restock calculations using sales rate, stock level, and supplier lead time. Keep the visualizations, but explain what each chart is showing.

### Final Solution:

#### What changed from the generated code

**Data validation.** Every run now checks the input before any calculation:
- **Required columns:** all seven must be present. Headers are matched after trimming spaces and ignoring case, and an error names any that are missing.
- **Numeric columns:** text such as `"abc"` or `"N/A"`, infinite values and negative numbers are treated as missing. Thousands separators and `$` signs are stripped first, so `"1,200"` and `"$4.50"` still parse.
- **Lead times:** anything over `max_lead_time_days` (default 365) is treated as a data error.
- **Dates:** a date is rejected if it can't be parsed, is after the as-of date, or is before 2000 (usually a number misread as a date).
- **Product IDs:** rows with no `product_id` are dropped. For a duplicated ID, only the most recent restock record is kept.

The code never guesses a replacement for a bad value. It leaves the value blank and records the reason in a `data_issues` column. If stock, sales or lead time can't be used, the product is marked **Needs review** and no restock advice is calculated for it.

**Bugs found in the original code:**
1. **Order quantities were too large for urgent items.** The original formula was `daily × (lead + safety + 30) − stock`. That formula assumes customers wait for out-of-stock items (backorders). If a product sells out before the delivery arrives, those sales are lost, so the order was too large by the demand during the gap. In example C below, the original ordered 84 units where 74 is correct. Whether customers wait is now a setting (`backorders`).
2. **Products with no stock and no sales were labeled "Healthy".** Zero sales often just means there was nothing to sell. These products now get their own status: **Out of stock, no sales**.
3. **A missing lead time made products look healthy.** Comparisons with a missing value are always False, so the restock checks silently failed.
4. **Turnover for products with no stock and no sales was set to 0.** Turnover is undefined there, so it is now left blank.
5. **Future restock dates produced negative "days since restock".**
6. **The chart styling changed every later chart in the notebook.** The style is now applied only to this figure.

#### Why these thresholds (all are settings in `InventoryConfig`)

**90-day slow-moving threshold.** 90 days of supply is about one quarter of stock, which works out to roughly 4 inventory turns a year (365 ÷ 90). That is a common rule of thumb for general merchandise, and many retailers review markdowns quarterly. The first version used it as a generic default. It was not tuned to this data. It is too loose for fast categories: groceries and other perishables should be flagged at 2 to 4 weeks. It may be too strict for slow, expensive goods. You can set a threshold for each category with `slow_moving_days_by_category`. A good value is each category's own normal level, for example the 75th percentile of days of supply.

**7-day safety buffer.** The code keeps one extra week of average sales on top of expected demand during the supplier lead time. This covers moderate sales spikes or a few days of supplier delay, and fits a weekly ordering routine. It is a flat rule of thumb because the data has only one 30-day sales total per product. The standard formula, safety stock = z × σ(daily demand) × √(lead time), needs daily sales history to measure how much demand varies. You can raise the buffer for unreliable suppliers or critical products with `safety_days_by_category`.

#### How the restock calculation works

| Quantity | Formula |
|---|---|
| Daily sales rate | `sales_last_30_days / 30` |
| Days of supply | `stock_level / daily sales` (infinite if there is stock but no sales) |
| Reorder point | `daily sales × (lead time + safety days)` |
| **Urgent restock** | days of supply < lead time: stock runs out before an order placed today arrives |
| **Restock soon** | stock ≤ reorder point: stock is already inside the safety buffer |
| Units left when the order arrives | `max(stock − daily sales × lead time, 0)`; lost sales can't make stock negative |
| Suggested order quantity | `daily sales × (safety days + coverage days) − units left when the order arrives` |
| Reorder-by date | as-of date + (days of supply − lead time − safety days) |

Worked examples, which the check cell below tests (7 safety days, 30 coverage days):

| | Stock | 30-day sales | Lead time | Days of supply | Reorder point | Status | Order |
|---|---|---|---|---|---|---|---|
| A | 100 | 60 (2/day) | 10 | 50 | 34 | Healthy | 0 |
| B | 30 | 60 | 10 | 15 | 34 | Restock soon | 2×37 − (30−20) = **64** |
| C | 10 | 60 | 10 | 5 | 34 | Urgent (5 days of stockout) | 2×37 − 0 = **74** |
| D | 0 | 60 | 10 | 0 | 34 | Urgent (10 days of stockout) | **74** |
| G | 100 | 30 (1/day) | 5 | 100 | 12 | Slow-moving | 0 |

**Known limits:**
- The data has no "quantity on order" column. A product that already has an order placed will still show as needing a restock.
- Turnover estimates opening stock as current stock plus units sold. That is wrong for products restocked during the last 30 days, which are marked in `restocked_in_window`.

In [ ]:
import numbers
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


@dataclass
class InventoryConfig:
    as_of_date: Optional[str] = None     # date the stock snapshot was taken; None = today
    sales_window_days: float = 30        # period covered by sales_last_30_days
    slow_moving_days: float = 90         # more days of supply than this = slow-moving
    safety_days: float = 7               # days of extra demand kept on top of lead-time demand
    order_coverage_days: float = 30      # days of demand a new order should cover once it arrives
    backorders: bool = False             # True if customers wait for out-of-stock items (sales aren't lost)
    max_lead_time_days: float = 365      # longer lead times are treated as data errors
    slow_moving_days_by_category: dict = field(default_factory=dict)  # e.g. {"Grocery": 21}
    safety_days_by_category: dict = field(default_factory=dict)       # e.g. {"Electronics": 14}

    def __post_init__(self):
        for name in ("sales_window_days", "slow_moving_days", "order_coverage_days", "max_lead_time_days"):
            if not self._is_number(getattr(self, name)) or getattr(self, name) <= 0:
                raise ValueError(f"{name} must be a number > 0, got {getattr(self, name)!r}")
        if not self._is_number(self.safety_days) or self.safety_days < 0:
            raise ValueError(f"safety_days must be a number >= 0, got {self.safety_days!r}")
        bad = {k: v for k, v in self.slow_moving_days_by_category.items()
               if not self._is_number(v) or v <= 0}
        bad.update({k: v for k, v in self.safety_days_by_category.items()
                    if not self._is_number(v) or v < 0})
        if bad:
            raise ValueError(f"Per-category thresholds must be numbers (slow > 0, safety >= 0): {bad}")
        self.as_of  # fail now, not mid-analysis, if as_of_date can't be parsed

    @staticmethod
    def _is_number(v):
        return isinstance(v, numbers.Real) and not isinstance(v, bool) and np.isfinite(v)

    @property
    def as_of(self):
        ts = pd.Timestamp.today() if self.as_of_date is None else pd.Timestamp(self.as_of_date)
        if ts.tzinfo is not None:
            ts = ts.tz_convert(None)
        return ts.normalize()


# Change thresholds here, e.g.
# CONFIG = InventoryConfig(slow_moving_days=60, slow_moving_days_by_category={"Grocery": 21})
CONFIG = InventoryConfig()

In [ ]:
# --- Sample data: replace this cell with your own DataFrame ---
rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "product_id": [f"P{i:04d}" for i in range(1, n + 1)],
    "category": rng.choice(["Electronics", "Apparel", "Grocery", "Home", "Toys"], n),
    "stock_level": rng.integers(0, 500, n),
    "last_restock_date": CONFIG.as_of - pd.to_timedelta(rng.integers(1, 180, n), unit="D"),
    "sales_last_30_days": rng.poisson(rng.gamma(1.5, 40, n)),
    "supplier_lead_time": rng.integers(3, 30, n),  # days
    "unit_cost": rng.uniform(2, 250, n).round(2),
})

# A few deliberately bad rows to show the validation working
dirty = pd.DataFrame({
    "product_id": ["P0001", "P9001", "P9002", "P9003", "P9004", None, "P9005"],
    "category": ["Toys", None, "Home", "Grocery", "Apparel", "Home", "Electronics"],
    "stock_level": [50, -12, "abc", "1,200", 25, 10, 0],
    "last_restock_date": [CONFIG.as_of, "not a date", "2026-08-15", "2099-01-01", None, "2026-09-01", "2026-06-01"],
    "sales_last_30_days": [20, 15, 30, np.nan, 12, 5, 0],
    "supplier_lead_time": [10, 7, 14, 5, -3, 7, 21],
    "unit_cost": [9.99, 4.5, 12.0, "$3.25", "N/A", 8.0, 30.0],
})
df = pd.concat([df, dirty], ignore_index=True)
df.tail(8)

In [ ]:
REQUIRED_COLUMNS = ["product_id", "category", "stock_level", "last_restock_date",
                    "sales_last_30_days", "supplier_lead_time", "unit_cost"]
NUMERIC_COLUMNS = ["stock_level", "sales_last_30_days", "supplier_lead_time", "unit_cost"]
EARLIEST_VALID_DATE = pd.Timestamp("2000-01-01")  # earlier dates are usually numbers misread as dates

REVIEW, URGENT, SOON, OUT_NO_SALES, SLOW, HEALTHY = (
    "Needs review", "Urgent restock", "Restock soon", "Out of stock, no sales", "Slow-moving", "Healthy")


def validate_columns(df):
    """Check the input is a non-empty DataFrame with every required column; return a copy with tidy headers."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(df).__name__}")
    df = df.rename(columns=lambda c: str(c).strip().lower())
    if df.columns.duplicated().any():
        raise ValueError(f"Duplicate column names after tidying headers: {list(df.columns[df.columns.duplicated()])}")
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}. Found: {list(df.columns)}")
    if df.empty:
        raise ValueError("The DataFrame has no rows")
    return df


def _to_float(s):
    """Parse a column to float64. Unparseable values and +/-inf become NaN."""
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype(str).str.replace(r"[,$\s]", "", regex=True)  # "1,200" and "$4.50" still parse
    num = pd.to_numeric(s, errors="coerce")
    num = pd.Series(num.to_numpy(dtype="float64", na_value=np.nan), index=s.index)
    return num.replace([np.inf, -np.inf], np.nan)


def _to_datetime(s):
    """Parse a column to naive datetimes. Unparseable values become NaT."""
    try:
        d = pd.to_datetime(s, errors="coerce", utc=True, format="mixed")
    except (TypeError, ValueError):  # pandas < 2.0 has no format="mixed"
        d = pd.to_datetime(s, errors="coerce", utc=True)
    return d.dt.tz_localize(None)


def _to_text(s):
    return s.astype("string").str.strip().fillna("").astype(object)


def clean_inventory(df, config=CONFIG):
    """Validate and clean raw inventory data.

    Invalid values become NaN/NaT (never guessed) and each row's problems are listed in `data_issues`.
    Returns (clean, dropped): rows with no product_id and older duplicate records go to `dropped`.
    """
    out = validate_columns(df)[REQUIRED_COLUMNS].copy()
    issues = {}

    def flag(label, mask):
        issues[label] = pd.Series(mask, index=out.index).fillna(False).astype(bool)

    out["product_id"] = _to_text(out["product_id"])
    flag("product_id: missing", out["product_id"] == "")

    out["category"] = _to_text(out["category"])
    flag("category: missing (set to Unknown)", out["category"] == "")
    out.loc[out["category"] == "", "category"] = "Unknown"

    for col in NUMERIC_COLUMNS:
        num = _to_float(out[col])
        flag(f"{col}: missing or not a number", num.isna())
        flag(f"{col}: negative", num < 0)
        out[col] = num.mask(num < 0)

    too_long = out["supplier_lead_time"] > config.max_lead_time_days
    flag(f"supplier_lead_time: over {config.max_lead_time_days:g} days", too_long)
    out["supplier_lead_time"] = out["supplier_lead_time"].mask(too_long)

    dates = _to_datetime(out["last_restock_date"])
    flag("last_restock_date: missing or unparseable", dates.isna())
    bad_date = (dates.dt.normalize() > config.as_of) | (dates < EARLIEST_VALID_DATE)
    flag("last_restock_date: in the future or before 2000", bad_date)
    out["last_restock_date"] = dates.mask(bad_date)

    flags = pd.DataFrame(issues)
    out["data_issues"] = ["; ".join(flags.columns[row]) for row in flags.to_numpy()]

    # Rows without an ID can't be reported on; for duplicate IDs keep the most recent restock record.
    no_id = out["product_id"] == ""
    dropped = [out[no_id].assign(drop_reason="missing product_id")]
    out = out[~no_id].sort_values("last_restock_date", ascending=False, na_position="last", kind="stable")
    dup = out.duplicated("product_id", keep="first")
    dropped.append(out[dup].assign(drop_reason="duplicate product_id (older record)"))
    dup_ids = out.loc[dup, "product_id"]
    out = out[~dup].sort_index()
    had_dup = out["product_id"].isin(dup_ids)
    note = "duplicate product_id: kept most recent record"
    out.loc[had_dup, "data_issues"] = out.loc[had_dup, "data_issues"].map(lambda s: f"{s}; {note}" if s else note)

    if out.empty:
        raise ValueError("No usable rows left after validation")
    return out, pd.concat(dropped)


def analyze_inventory(df, config=CONFIG):
    """Clean the data, then add turnover, days of supply, slow-moving and restock columns.

    Returns (inv, dropped).
    """
    inv, dropped = clean_inventory(df, config)
    stock, sales, lead = inv["stock_level"], inv["sales_last_30_days"], inv["supplier_lead_time"]
    window = config.sales_window_days

    slow_days = inv["category"].map(config.slow_moving_days_by_category).astype(float).fillna(config.slow_moving_days)
    safety = inv["category"].map(config.safety_days_by_category).astype(float).fillna(config.safety_days)

    daily = sales / window
    inv["daily_sales"] = daily
    inv["inventory_value"] = stock * inv["unit_cost"]  # NaN if unit_cost is invalid
    inv["days_since_restock"] = (config.as_of - inv["last_restock_date"]).dt.days
    inv["restocked_in_window"] = inv["days_since_restock"] < window  # turnover estimate less reliable

    # Turnover = COGS / average inventory value. unit_cost appears on both sides and cancels,
    # so it's computed in units and a bad unit_cost doesn't block it.
    # Opening stock is estimated as current stock + units sold.
    avg_units = stock + sales / 2
    inv["turnover_period"] = sales / avg_units.where(avg_units > 0)  # NaN when nothing held or sold
    inv["turnover_annual"] = inv["turnover_period"] * 365 / window

    # Days of supply: stock with no sales never runs out (inf); no stock and no sales = 0.
    dos = stock / daily.where(daily > 0)
    dos = dos.mask((daily == 0) & (stock > 0), np.inf).mask((daily == 0) & (stock == 0), 0.0)
    inv["days_of_supply"] = dos
    inv["slow_moving_threshold"] = slow_days
    inv["safety_days"] = safety

    # Restock status. Rows missing stock, sales or lead time can't be assessed.
    assessable = stock.notna() & sales.notna() & lead.notna()
    selling = assessable & (daily > 0)
    inv["reorder_point"] = daily * (lead + safety)
    urgent = selling & (dos < lead)                                      # runs out before an order placed today arrives
    soon = selling & ~urgent & (stock <= inv["reorder_point"])          # inside the safety buffer
    out_no_sales = assessable & (stock == 0) & (daily == 0)             # demand unknown: nothing to sell
    slow = assessable & (dos > slow_days)
    inv["status"] = np.select([~assessable, urgent, soon, out_no_sales, slow],
                              [REVIEW, URGENT, SOON, OUT_NO_SALES, SLOW], default=HEALTHY)

    # Order quantity: enough to cover safety + coverage days once the order arrives.
    # With lost sales, stock bottoms out at 0 while waiting; with backorders the shortfall is owed too.
    left_at_arrival = stock - daily * lead
    if not config.backorders:
        left_at_arrival = left_at_arrival.clip(lower=0)
    target_at_arrival = daily * (safety + config.order_coverage_days)
    qty = np.ceil((target_at_arrival - left_at_arrival).clip(lower=0).round(6))  # round() stops 64.0000001 -> 65
    inv["suggested_order_qty"] = qty.where(urgent | soon, 0).mask(~assessable).astype("Int64")

    inv["stockout_days_before_arrival"] = (lead - dos).clip(lower=0).where(selling)
    reorder_in = (dos - (lead + safety)).clip(lower=0).where(assessable)
    reorder_in = np.floor(reorder_in.where(np.isfinite(reorder_in)))  # inf (never runs out) -> no date
    inv["reorder_by"] = config.as_of + pd.to_timedelta(reorder_in, unit="D")

    return inv, dropped

In [ ]:
# Restock-logic check: hand-worked cases from the table above. Fixed settings so changing CONFIG doesn't break it.
check_cfg = InventoryConfig(as_of_date="2026-01-31", safety_days=7, order_coverage_days=30, slow_moving_days=90)
cases = pd.DataFrame({
    "product_id":         ["A", "B", "C", "D", "E", "F", "G", "H"],
    "category":           "Test",
    "stock_level":        [100, 30, 10, 0, 200, 0, 100, -5],
    "last_restock_date":  "2026-01-01",
    "sales_last_30_days": [60, 60, 60, 60, 0, 0, 30, 60],
    "supplier_lead_time": [10, 10, 10, 10, 10, 10, 5, 10],
    "unit_cost":          10.0,
})
expected = pd.DataFrame({
    "status": [HEALTHY, SOON, URGENT, URGENT, SLOW, OUT_NO_SALES, SLOW, REVIEW],
    "days_of_supply": [50, 15, 5, 0, np.inf, 0, 100, np.nan],
    "suggested_order_qty": [0, 64, 74, 74, 0, 0, 0, np.nan],
    "stockout_days_before_arrival": [0, 0, 5, 10, np.nan, np.nan, 0, np.nan],
}, index=list("ABCDEFGH"))

got = analyze_inventory(cases, check_cfg)[0].set_index("product_id")
assert (got["status"] == expected["status"]).all(), got["status"]
for col in ["days_of_supply", "suggested_order_qty", "stockout_days_before_arrival"]:
    np.testing.assert_allclose(got[col].astype(float), expected[col], err_msg=col)
assert np.isclose(got.loc["A", "turnover_annual"], 60 / (100 + 30) * 365 / 30)  # about 5.6x a year

# With backorders, C's order also covers the 10 units of demand it misses while out of stock
got_bo = analyze_inventory(cases, InventoryConfig(as_of_date="2026-01-31", backorders=True))[0]
assert got_bo.set_index("product_id").loc["C", "suggested_order_qty"] == 84

# Missing columns are reported by name
try:
    analyze_inventory(cases.drop(columns=["unit_cost", "category"]), check_cfg)
except ValueError as e:
    print("Column check works:", e)
else:
    raise AssertionError("Missing columns were not detected")

print("All restock checks passed.")

In [ ]:
inv, dropped = analyze_inventory(df, CONFIG)

print(f"As of {CONFIG.as_of:%Y-%m-%d}: {len(df)} rows in, {len(inv)} analyzed, {len(dropped)} dropped")
if not dropped.empty:
    display(dropped[["product_id", "drop_reason", "data_issues"]])

flagged = inv[inv["data_issues"] != ""]
print(f"\nRows with data issues: {len(flagged)} ({(inv['status'] == REVIEW).sum()} need review before any restock advice)")
display(flagged[["product_id", "status", "data_issues"]])

print("\nProducts by status:")
print(inv["status"].value_counts().to_string())

print("\nMedian annual turnover by category:")
print(inv.groupby("category")["turnover_annual"].median().dropna().sort_values().round(2).to_string())

slow_items = (inv[inv["status"] == SLOW]
              .sort_values("inventory_value", ascending=False, na_position="last")
              [["product_id", "category", "stock_level", "sales_last_30_days", "days_of_supply",
                "slow_moving_threshold", "turnover_annual", "inventory_value", "days_since_restock"]])
print(f"\nSlow-moving items: {len(slow_items)}, "
      f"${slow_items['inventory_value'].sum():,.0f} of inventory tied up")
display(slow_items.head(10).round(1))

restock = (inv[inv["status"].isin([URGENT, SOON])]
           .assign(days_buffer=lambda d: d["days_of_supply"] - d["supplier_lead_time"])
           .sort_values("days_buffer")
           [["product_id", "category", "status", "stock_level", "daily_sales", "days_of_supply",
             "supplier_lead_time", "reorder_point", "stockout_days_before_arrival",
             "suggested_order_qty", "reorder_by"]])
print(f"\nProducts to restock: {len(restock)} "
      f"({(restock['status'] == URGENT).sum()} will run out before a new order arrives)")
display(restock.head(15).round(1))

out_no_sales = inv[inv["status"] == OUT_NO_SALES]
if not out_no_sales.empty:
    print(f"\nOut of stock with no recent sales ({len(out_no_sales)}), check whether they're still stocked:")
    display(out_no_sales[["product_id", "category", "last_restock_date", "supplier_lead_time"]])

#### What each chart shows

1. **Days of supply vs. supplier lead time (top left).** Each dot is one product. The horizontal position is how long the supplier takes to deliver. The vertical position is how many days current stock lasts at the current sales rate.
   - **Dots below the dashed diagonal** will run out before an order placed today could arrive. These are the urgent restocks.
   - **Dots above the dotted line** have more stock than the slow-moving threshold.
   - The vertical scale stops at 180 days, so products with stock but no sales sit along the top edge.
   - Products marked "Needs review" are left out because their position can't be calculated.
2. **Inventory turnover by category (top right).** This shows the median number of times a year each category sells through its stock. A higher number means faster-moving stock. The median is used so a few extreme products don't distort a category. About 4× a year is the same as 90 days of supply.
3. **Where inventory value sits (bottom left).** This shows the value of stock on hand (stock × unit cost) in each category, split by status. The gray segment is money tied up in slow-moving products, which are candidates for markdowns or smaller orders. Products with an invalid unit cost are left out.
4. **Top 15 restock priorities (bottom right).** Products are ordered by the smallest gap between days of supply and lead time. Each bar shows how many days until the product runs out. The black tick marks the supplier lead time. If a bar ends to the left of its tick, the product will be out of stock for that many days even if it is reordered today.

In [ ]:
STATUS_COLORS = {
    URGENT: "#d03b3b",
    SOON: "#fab219",
    OUT_NO_SALES: "#ec835a",
    SLOW: "#898781",
    HEALTHY: "#0ca30c",
    REVIEW: "#c3c2b7",
}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
DOS_CAP = 180  # days of supply above this (including inf) are drawn at the cap
STYLE = {
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
}


def _empty(ax, title, message):
    ax.set_title(title)
    ax.text(0.5, 0.5, message, ha="center", va="center", color=MUTED, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])


with plt.rc_context(STYLE):  # style only this figure, not the rest of the notebook
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    ax1, ax2, ax3, ax4 = axes.flat

    # (1) Days of supply vs supplier lead time
    plotted = inv[inv["days_of_supply"].notna() & inv["supplier_lead_time"].notna()]
    for status, color in STATUS_COLORS.items():
        d = plotted[plotted["status"] == status]
        if not d.empty:
            ax1.scatter(d["supplier_lead_time"], d["days_of_supply"].clip(upper=DOS_CAP),
                        s=36, color=color, edgecolor="white", linewidth=0.8, label=f"{status} ({len(d)})")
    lt = np.array([0, plotted["supplier_lead_time"].max() + 2 if not plotted.empty else 30])
    ax1.plot(lt, lt, color=INK, lw=1, ls="--")
    ax1.text(lt[1], lt[1] + 4, "runs out before\nreorder arrives", ha="right", va="bottom", fontsize=8, color=MUTED)
    slow_label = "default " if CONFIG.slow_moving_days_by_category else ""
    ax1.axhline(CONFIG.slow_moving_days, color=MUTED, lw=1, ls=":")
    ax1.text(lt[1], CONFIG.slow_moving_days + 3,
             f"{slow_label}slow-moving threshold ({CONFIG.slow_moving_days:g} days)",
             ha="right", fontsize=8, color=MUTED)
    ax1.set(xlabel="Supplier lead time (days)", ylabel=f"Days of supply (capped at {DOS_CAP})",
            title="Days of supply vs. supplier lead time", ylim=(-5, DOS_CAP + 10))
    ax1.grid(color=GRID, lw=0.6)
    ax1.legend(frameon=False, loc="upper left", fontsize=8)

    # (2) Median annual turnover by category
    turn = inv.groupby("category")["turnover_annual"].median().dropna().sort_values()
    if turn.empty:
        _empty(ax2, "Inventory turnover by category", "No products with a valid turnover")
    else:
        ax2.barh(turn.index, turn.values, color="#2a78d6", height=0.6)
        for y, v in enumerate(turn.values):
            ax2.text(v, y, f" {v:.1f}×", va="center", color=INK)
        ax2.set(xlabel="Median inventory turnover (times per year)", title="Inventory turnover by category")
        ax2.grid(axis="x", color=GRID, lw=0.6)

    # (3) Inventory value by category, split by status
    valued = inv[inv["inventory_value"].notna()]
    if valued.empty:
        _empty(ax3, "Where inventory value sits", "No products with a valid unit cost")
    else:
        value = (valued.pivot_table(index="category", columns="status", values="inventory_value",
                                    aggfunc="sum", fill_value=0)
                       .reindex(columns=list(STATUS_COLORS), fill_value=0))
        value = value.loc[value.sum(axis=1).sort_values().index]
        left = np.zeros(len(value))
        for status, color in STATUS_COLORS.items():
            if value[status].sum() > 0:
                ax3.barh(value.index, value[status] / 1000, left=left, color=color,
                         edgecolor="white", linewidth=1.5, height=0.6, label=status)
                left += value[status].to_numpy() / 1000
        ax3.set(xlabel="Inventory value ($ thousands)", title="Where inventory value sits")
        ax3.grid(axis="x", color=GRID, lw=0.6)
        ax3.legend(frameon=False, loc="lower right", fontsize=8)

    # (4) Top 15 restock priorities
    if restock.empty:
        _empty(ax4, "Top 15 restock priorities", "No products need restocking")
    else:
        top = restock.head(15).iloc[::-1]
        ax4.barh(top["product_id"], top["days_of_supply"], height=0.6, color=top["status"].map(STATUS_COLORS))
        ax4.scatter(top["supplier_lead_time"], top["product_id"], marker="|", s=250, color=INK,
                    linewidth=2, label="Supplier lead time", zorder=3)
        ax4.set(xlabel="Days", title="Top 15 restock priorities (bar = days until stock runs out)")
        ax4.grid(axis="x", color=GRID, lw=0.6)
        ax4.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=STATUS_COLORS[s]) for s in (URGENT, SOON)]
                           + ax4.get_legend_handles_labels()[0],
                   labels=[URGENT, SOON, "Supplier lead time"], frameon=False, loc="lower right", fontsize=8)

    fig.suptitle(f"Retail inventory health, as of {CONFIG.as_of:%Y-%m-%d}",
                 fontsize=15, fontweight="bold", x=0.01, ha="left", color=INK)
    fig.tight_layout()
    plt.show()